# Validation

Three checks: baseline comparison (k-means, agglomerative at k=107),
NPMI lexical coherence, sensitivity analysis (±20% perturbations).

In [2]:
import numpy as np
import pandas as pd
import json
import re
import warnings
from collections import Counter, defaultdict
from itertools import combinations
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import umap
import hdbscan

warnings.filterwarnings("ignore")

## 1. Load

In [3]:
df = pd.read_csv("tovima_final.csv", encoding="utf-32", sep="\t")
embeddings_raw = np.load("tovima_embeddings_bge_m3.npy")

assert len(df) == len(embeddings_raw), "Alignment error"

embeddings = normalize(embeddings_raw, norm="l2")
final_labels = df["final_label"].values

print(f"Rows: {len(df)}")
print(f"Embedding dim: {embeddings.shape[1]}")
print(f"HDBSCAN clusters: {len(set(final_labels) - {-1})}")
print(f"Noise points: {(final_labels == -1).sum()} ({(final_labels == -1).mean()*100:.1f}%)")

Rows: 13637
Embedding dim: 1024
HDBSCAN clusters: 107
Noise points: 2011 (14.7%)


In [4]:
# PCA, same settings as clustering.ipynb
pca = PCA(n_components=256, svd_solver="randomized", random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
print(f"PCA: 256 components, {pca.explained_variance_ratio_.sum()*100:.1f}% variance retained")

PCA: 256 components, 87.7% variance retained


## 2. Semantic coherence function

In [5]:
def semantic_coherence(embeddings, labels, min_cluster_size=10):
    cluster_ids = set(labels) - {-1}
    scores, weights = [], []
    for c in cluster_ids:
        idx = np.where(labels == c)[0]
        if len(idx) < min_cluster_size:
            continue
        vecs = embeddings[idx]
        centroid = vecs.mean(axis=0, keepdims=True)
        sims = cosine_similarity(vecs, centroid).ravel()
        scores.append(np.median(sims))
        weights.append(len(idx))
    if not scores:
        return 0.0
    return float(np.average(scores, weights=weights))

# HDBSCAN score on final labels
hdbscan_coherence = semantic_coherence(embeddings, final_labels)
print(f"HDBSCAN semantic coherence: {hdbscan_coherence:.4f}")

HDBSCAN semantic coherence: 0.8111


## 3. Baseline 1: k-means (k=107)

In [6]:
N_CLUSTERS = len(set(final_labels) - {-1})
print(f"Running k-means with k={N_CLUSTERS}...")

km = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=5,
    max_iter=300,
)
km_labels = km.fit_predict(embeddings_pca)
km_coherence = semantic_coherence(embeddings, km_labels)

print(f"k-means semantic coherence: {km_coherence:.4f}")
print(f"(HDBSCAN: {hdbscan_coherence:.4f}, delta: {hdbscan_coherence - km_coherence:+.4f})")

Running k-means with k=107...
k-means semantic coherence: 0.8009
(HDBSCAN: 0.8111, delta: +0.0102)


## 4. Baseline 2: agglomerative clustering (k=107)

Full distance matrix ~1.5 GB; if it OOMs, sample down.

In [7]:
print(f"Running agglomerative clustering (Ward, k={N_CLUSTERS})...")
print("This may take a few minutes...")

agg = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    linkage="ward",
)
agg_labels = agg.fit_predict(embeddings_pca)
agg_coherence = semantic_coherence(embeddings, agg_labels)

print(f"Agglomerative coherence: {agg_coherence:.4f}")
print(f"(HDBSCAN: {hdbscan_coherence:.4f}, delta: {hdbscan_coherence - agg_coherence:+.4f})")

Running agglomerative clustering (Ward, k=107)...
This may take a few minutes...
Agglomerative coherence: 0.8005
(HDBSCAN: 0.8111, delta: +0.0106)


## 5. Baseline comparison summary

In [8]:
results = pd.DataFrame([
    {"method": "HDBSCAN (our pipeline)", "n_clusters": N_CLUSTERS,
     "noise_pct": round((final_labels == -1).mean() * 100, 1),
     "semantic_coherence": round(hdbscan_coherence, 4)},
    {"method": "k-means (k=107)", "n_clusters": N_CLUSTERS,
     "noise_pct": 0.0,
     "semantic_coherence": round(km_coherence, 4)},
    {"method": "Agglomerative Ward (k=107)", "n_clusters": N_CLUSTERS,
     "noise_pct": 0.0,
     "semantic_coherence": round(agg_coherence, 4)},
])
print(results.to_string(index=False))
print()
print("Note: noise_pct = 0 for partition methods since they assign every point.")
print("HDBSCAN's noise points are genuinely borderline emails that don't cluster")
print("with anything — this is a feature, not a deficit.")

                    method  n_clusters  noise_pct  semantic_coherence
    HDBSCAN (our pipeline)         107       14.7              0.8111
           k-means (k=107)         107        0.0              0.8009
Agglomerative Ward (k=107)         107        0.0              0.8005

Note: noise_pct = 0 for partition methods since they assign every point.
HDBSCAN's noise points are genuinely borderline emails that don't cluster
with anything — this is a feature, not a deficit.


## 6. NPMI coherence

Computed on `cleaned_text`, independent of the embeddings.
Negative values are normal for email corpora; only the ranking across methods matters.

In [9]:
def build_cooccurrence(texts, labels, top_n_terms=10, window="doc"):
    """
    Build global term document frequencies only — not all pairwise counts,
    which would require O(vocab^2) memory. Per-cluster pair frequencies are
    computed locally in cluster_npmi, touching only the top-N terms per
    cluster rather than the full vocabulary.
    """
    term_freq = Counter()
    doc_count = 0
    for text, label in zip(texts, labels):
        if label == -1 or not isinstance(text, str):
            continue
        tokens = {t for t in text.split() if len(t) >= 4}
        if not tokens:
            continue
        doc_count += 1
        for t in tokens:
            term_freq[t] += 1
    return term_freq, doc_count


def cluster_npmi(texts, labels, top_n=10, min_cluster_size=10):
    """
    For each cluster: find the top-N terms by within-cluster frequency,
    compute mean pairwise NPMI using corpus-wide term frequencies and
    within-cluster pair frequencies (computed locally per cluster to
    avoid storing all pairs globally).
    """
    # global term frequencies
    term_freq, doc_count = build_cooccurrence(texts, labels)

    cluster_ids = set(labels) - {-1}
    scores, weights = [], []

    for c in cluster_ids:
        cluster_texts = [t for t, l in zip(texts, labels)
                         if l == c and isinstance(t, str)]
        if len(cluster_texts) < min_cluster_size:
            continue

        # top-N terms per cluster
        cluster_tf = Counter()
        pair_freq = Counter()
        for text in cluster_texts:
            tokens = {t for t in text.split() if len(t) >= 4}
            for t in tokens:
                cluster_tf[t] += 1
            # pairs within this document
            for pair in combinations(sorted(tokens), 2):
                pair_freq[pair] += 1

        top_terms = [t for t, _ in cluster_tf.most_common(top_n)]
        if len(top_terms) < 2:
            continue

        # mean pairwise NPMI
        pair_scores = []
        for t1, t2 in combinations(top_terms, 2):
            p1 = term_freq[t1] / doc_count
            p2 = term_freq[t2] / doc_count
            pp = pair_freq.get((min(t1, t2), max(t1, t2)), 0) / doc_count
            if pp == 0 or p1 == 0 or p2 == 0:
                pair_scores.append(-1.0)
                continue
            pmi = np.log(pp / (p1 * p2))
            npmi = pmi / (-np.log(pp))
            pair_scores.append(float(npmi))

        if pair_scores:
            scores.append(np.mean(pair_scores))
            weights.append(len(cluster_texts))

    if not scores:
        return 0.0
    return float(np.average(scores, weights=weights))

In [10]:
texts = df["cleaned_text"].tolist()

print("Computing NPMI for HDBSCAN labels...")
hdbscan_npmi = cluster_npmi(texts, final_labels.tolist())
print(f"HDBSCAN NPMI: {hdbscan_npmi:.4f}")

print("Computing NPMI for k-means labels...")
km_npmi = cluster_npmi(texts, km_labels.tolist())
print(f"k-means NPMI: {km_npmi:.4f}")

print("Computing NPMI for agglomerative labels...")
agg_npmi = cluster_npmi(texts, agg_labels.tolist())
print(f"Agglomerative NPMI: {agg_npmi:.4f}")

Computing NPMI for HDBSCAN labels...
HDBSCAN NPMI: -0.2077
Computing NPMI for k-means labels...
k-means NPMI: -0.2979
Computing NPMI for agglomerative labels...
Agglomerative NPMI: -0.2893


## 7. Full comparison table

In [11]:
results["npmi_coherence"] = [
    round(hdbscan_npmi, 4),
    round(km_npmi, 4),
    round(agg_npmi, 4),
]
print(results.to_string(index=False))

                    method  n_clusters  noise_pct  semantic_coherence  npmi_coherence
    HDBSCAN (our pipeline)         107       14.7              0.8111         -0.2077
           k-means (k=107)         107        0.0              0.8009         -0.2979
Agglomerative Ward (k=107)         107        0.0              0.8005         -0.2893


## 8. Sensitivity analysis

±20% around best_params.json, one parameter at a time.

In [12]:
with open("best_params.json") as f:
    best = json.load(f)

bp = best["best_params"]
print("Best params from Optuna:")
for k, v in bp.items():
    print(f"  {k}: {v}")
print(f"Best score: {best['best_value']:.4f}")
print(f"Best attrs: {best['best_attrs']}")

Best params from Optuna:
  n_neighbors: 11
  n_components: 13
  min_dist: 0.014138059874308344
  min_cluster_size: 32
  min_samples: 4
Best score: 0.8081
Best attrs: {'n_clusters': 116, 'noise_pct': 15.45, 'coherence': 0.8126, 'stability': 0.0892, 'median_cluster_size': 69.5, 'score': 0.8081}


In [13]:
def run_umap_hdbscan(embeddings_pca, embeddings_orig, params):
    reducer = umap.UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=params["n_components"],
        min_dist=params["min_dist"],
        metric="cosine",
        random_state=42,
    )
    reduced = reducer.fit_transform(embeddings_pca)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(reduced)
    n_clusters = len(set(labels) - {-1})
    noise_pct = float(np.mean(labels == -1)) * 100
    coherence = semantic_coherence(embeddings_orig, labels)
    return n_clusters, noise_pct, coherence


# ±20% per parameter
sensitivity_rows = []

# baseline
n_c, n_p, coh = run_umap_hdbscan(embeddings_pca, embeddings, bp)
sensitivity_rows.append({
    "perturbation": "baseline (best params)",
    "n_clusters": n_c, "noise_pct": round(n_p, 1), "coherence": round(coh, 4)
})
print(f"Baseline: clusters={n_c}, noise={n_p:.1f}%, coherence={coh:.4f}")

for param in ["n_neighbors", "n_components", "min_cluster_size", "min_samples"]:
    for direction, factor in [("−20%", 0.8), ("+20%", 1.2)]:
        perturbed = dict(bp)
        new_val = max(2, round(bp[param] * factor))
        perturbed[param] = new_val
        label = f"{param} {direction} ({bp[param]}→{new_val})"
        try:
            n_c, n_p, coh = run_umap_hdbscan(embeddings_pca, embeddings, perturbed)
            sensitivity_rows.append({
                "perturbation": label,
                "n_clusters": n_c, "noise_pct": round(n_p, 1), "coherence": round(coh, 4)
            })
            print(f"{label}: clusters={n_c}, noise={n_p:.1f}%, coherence={coh:.4f}")
        except Exception as e:
            print(f"{label}: FAILED — {e}")

sensitivity_df = pd.DataFrame(sensitivity_rows)
print()
print(sensitivity_df.to_string(index=False))

coherence_vals = sensitivity_df["coherence"].values
print(f"\nCoherence range across perturbations: {coherence_vals.min():.4f} – {coherence_vals.max():.4f}")
print(f"Std dev: {coherence_vals.std():.4f}")
print(f"Max deviation from baseline: {abs(coherence_vals - coherence_vals[0]).max():.4f}")

Baseline: clusters=116, noise=15.5%, coherence=0.8126
n_neighbors −20% (11→9): clusters=132, noise=19.3%, coherence=0.8163
n_neighbors +20% (11→13): clusters=123, noise=17.3%, coherence=0.8148
n_components −20% (13→10): clusters=129, noise=17.4%, coherence=0.8152
n_components +20% (13→16): clusters=128, noise=18.5%, coherence=0.8169
min_cluster_size −20% (32→26): clusters=154, noise=18.4%, coherence=0.8189
min_cluster_size +20% (32→38): clusters=98, noise=18.0%, coherence=0.8102
min_samples −20% (4→3): clusters=128, noise=19.3%, coherence=0.8159
min_samples +20% (4→5): clusters=118, noise=17.2%, coherence=0.8150

                 perturbation  n_clusters  noise_pct  coherence
       baseline (best params)         116       15.5     0.8126
      n_neighbors −20% (11→9)         132       19.3     0.8163
     n_neighbors +20% (11→13)         123       17.3     0.8148
    n_components −20% (13→10)         129       17.4     0.8152
    n_components +20% (13→16)         128       18.5     0.

## 9. Summary

In [14]:
print(results[["method", "semantic_coherence", "npmi_coherence"]].to_string(index=False))
print()
print(f"coherence range: {coherence_vals.min():.4f} - {coherence_vals.max():.4f}")
print(f"std across perturbations: {coherence_vals.std():.4f}")

                    method  semantic_coherence  npmi_coherence
    HDBSCAN (our pipeline)              0.8111         -0.2077
           k-means (k=107)              0.8009         -0.2979
Agglomerative Ward (k=107)              0.8005         -0.2893

coherence range: 0.8102 - 0.8189
std across perturbations: 0.0024
